In [0]:
with opps as (SELECT 
    o.id                           AS opp_id,
    ol.id                          AS oli_id,
    ql.pathfinder_enabled_c        AS pathfinder_enabled_ql,
    ol.pathfinder_enabled_c        AS pathfinder_enabled_ol,
    ql.id                          AS qli_id,
    ol.quantity as quantity    ,
    ol.quote_line_acv_c as quote_acv , 
    ql.transaction_type_c,
    o.type as opp_type,
    o.account_id                   AS account_id,
    CAST(o.close_date AS DATE)     AS close_date,
    p.product_code                 AS product_code,
    pp.product_code          AS base_product_code,
    p.family                       AS product_family,
    case when qr.sfdc_id is not null or r.sfdc_id is not null then 'reversal' else null end as reversal_opp
FROM 
    dev_dm.salesforce_bronze.opportunity o
LEFT JOIN 
    dev_dm.salesforce_bronze.opportunity_line_item ol ON ol.opportunity_id = o.id
LEFT JOIN 
    dev_dm.salesforce_bronze.sbqq_quote_line_c ql ON ol.sbqq_quote_line_c = ql.id
LEFT JOIN 
    dev_dm.salesforce_bronze.sbqq_quote_c q ON ql.sbqq_quote_c = q.id
LEFT JOIN 
    dev_dm.salesforce_bronze.product_2 p ON ol.product_2_id = p.id
LEFT JOIN
   (select distinct p.id, p.product_code FROM dev_dm.salesforce_bronze.product_2 p) pp ON p.base_product_c = pp.id
LEFT JOIN (select distinct sfdc_id from dev_dm.revops_analytics.reversals where link_type = 'Opportunity') r on o.id = r.sfdc_id
LEFT JOIN (select distinct sfdc_id from dev_dm.revops_analytics.reversals where link_type = 'Quote') qr on ql.id = qr.sfdc_id
WHERE 
    o.stage_name = '8 - Closed Won'
    AND o.close_date > '2022-12-31'
    AND q.sbqq_primary_c = 'true'
    AND ql.sbqq_charge_type_c = 'Recurring' -- No arr_type_c column found
    AND p.base_product_c IS NOT NULL
    AND p.arrclass_c = 'SaaS' -- No arr_class_c column found
    AND p.family IN (
        'Privileged Remote Access',
        'Password Safe',
        'Remote Support',
        'PM for Unix and Linux Servers',
        'PM for Desktops',
        'Entitle Platform',
        'PM for Windows Servers',
        'Workforce Passwords',
        'Password Safe with PRA',
        'Identity Security Insights'
    ))

SELECT *
FROM OPPS;